In [2]:
# Cài đặt các thư viện bắt buộc cho Pipeline và ép cập nhật torchao
!pip install -q -U "torchao>0.16.0"
!pip install -q open_clip_torch
!pip install -q evaluate
!pip install -q rouge_score
!pip install -q bert_score
!pip install -q scikit-learn
!pip install evaluate rouge_score bert_score scikit-learn
!pip install -q bitsandbytes accelerate
!pip install -q -U transformers accelerate datasets
!pip install -q einops
print("✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!")

✅ Đã cài đặt xong tất cả các thư viện cần thiết và cập nhật torchao!


In [3]:
import os
import gc
import time
import torch
import re
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from datasets import load_dataset
import transformers
from transformers import AutoModel, AutoTokenizer
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
import bitsandbytes as bnb 
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
import evaluate
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm 
import warnings
import urllib.request
import zipfile
import ctypes

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ==========================================
# 0. VÁ LỖI KIẾN TRÚC INTERNVL & HUGGINGFACE
# ==========================================
transformers.PreTrainedModel.all_tied_weights_keys = {}
transformers.logging.set_verbosity_error()
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
torch.backends.cudnn.benchmark = True 

NUM_CLIENTS = 2
runName = f"FedMVQA_SLAKE_InternVL3.5_FFT_448_Alpha1_R1_E2_{NUM_CLIENTS}Clients_OOMFix"
kaggleBaseDir = "/kaggle/working/Fed-MVQA-Project"
dataDir = os.path.join(kaggleBaseDir, "data", "slake")
logDir = os.path.join(kaggleBaseDir, "logs")
checkpointDir = os.path.join(kaggleBaseDir, "checkpoints", runName)
TEMP_WEIGHT_DIR = os.path.join(kaggleBaseDir, "temp_fed_weights")

os.makedirs(dataDir, exist_ok=True)
os.makedirs(logDir, exist_ok=True)
os.makedirs(checkpointDir, exist_ok=True)
os.makedirs(TEMP_WEIGHT_DIR, exist_ok=True)

logPath = os.path.join(logDir, f"{runName}_log.csv")
baselinePath = os.path.join(TEMP_WEIGHT_DIR, "global_baseline.pt")

# ==========================================
# 1. CÔNG CỤ QUẢN LÝ BỘ NHỚ
# ==========================================
def clearMemory():
    gc.collect()
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            torch.cuda.synchronize(i)
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except: pass

def getPeakReservedMemory():
    memory_info = []
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            memory_info.append(f"GPU{i}: {torch.cuda.memory_reserved(i)/(1024**3):.1f}GB")
    return " | ".join(memory_info)

def calculateCommunicationMetrics(totalParams, numClients):
    # Dựa trên float16 (2 bytes)
    peftBytes = totalParams * 2
    fullRoundMb = (peftBytes * 2 * numClients) / (1024 * 1024)
    return fullRoundMb

def processImage(pilImg, targetSize=448):
    if pilImg.mode != "RGB":
        pilImg = pilImg.convert("RGB")
    imageTransform = T.Compose([
        T.Resize((targetSize, targetSize), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])
    return imageTransform(pilImg).unsqueeze(0)

# ==========================================
# 2. KHỞI TẠO MÔ HÌNH TOÀN CỤC (INTERNVL3.5-2B FFT)
# ==========================================
print(f"--- KHỞI TẠO HẠ TẦNG FED-FFT (INTERNVL3.5-2B, 448x448, FP16 - {NUM_CLIENTS} CLIENTS) ---")
clearMemory()

modelId = "OpenGVLab/InternVL3_5-2B-Instruct"

# Sử dụng device_map="auto" để tự động chia model qua các GPU
baseModel = AutoModel.from_pretrained(
    modelId,
    torch_dtype=torch.float16, 
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(modelId, trust_remote_code=True)
print("✅ Đã nạp thành công mô hình InternVL3.5-2B!")

# ---------------------------------------------------------
# 🔥 BẢN VÁ LỖI MULTI-GPU CHO INTERNVL
if hasattr(baseModel, 'extract_feature'):
    original_extract = baseModel.extract_feature
    def safe_extract(*args, **kwargs):
        features = original_extract(*args, **kwargs)
        return features.to(baseModel.device)
    baseModel.extract_feature = safe_extract
# ---------------------------------------------------------

baseModel.config.use_cache = False
if hasattr(baseModel, 'enable_input_require_grads'): baseModel.enable_input_require_grads()
try: baseModel.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except: pass
if hasattr(baseModel, 'language_model'): baseModel.language_model.gradient_checkpointing_enable()
if hasattr(baseModel, 'vision_model'): baseModel.vision_model.gradient_checkpointing_enable()

# Mở khóa toàn bộ tham số cho FFT
for p in baseModel.parameters():
    p.requires_grad = True

trainableParams = sum(p.numel() for p in baseModel.parameters() if p.requires_grad)
totalModelParams = sum(p.numel() for p in baseModel.parameters())
print(f"[Metrics] Trainable Params (FFT): {trainableParams:,} (~100%)")

# Lưu trữ Baseline gốc trực tiếp xuống đĩa để giải phóng RAM
torch.save(baseModel.state_dict(), baselinePath)

# ==========================================
# 3. TRÍCH XUẤT ĐỘNG ID TOKEN HÌNH ẢNH 
# ==========================================
print(" Đang trích xuất tự động Image Context ID từ lõi mô hình...")
interceptedIds = None
originalForward = baseModel.forward

def hookForward(*args, **kwargs):
    global interceptedIds
    if 'input_ids' in kwargs and kwargs['input_ids'] is not None:
        interceptedIds = kwargs['input_ids'].detach().cpu()
    elif len(args) > 1 and args[1] is not None:
        interceptedIds = args[1].detach().cpu()
    raise RuntimeError("STOP_INTERCEPT")

baseModel.forward = hookForward
try:
    dummyPx = torch.zeros((1, 3, 448, 448), dtype=torch.float16).to(baseModel.device)
    baseModel.chat(tokenizer, dummyPx, "<image>", generation_config=dict(max_new_tokens=1))
except RuntimeError as e:
    if str(e) != "STOP_INTERCEPT":
        pass
except Exception:
    pass
finally:
    baseModel.forward = originalForward

if interceptedIds is not None:
    counts = torch.bincount(interceptedIds.view(-1))
    imgContextId = int(counts.argmax().item())
else:
    imgContextId = getattr(baseModel, 'img_context_token_id', tokenizer.convert_tokens_to_ids('<IMG_CONTEXT>'))
    
print(f"✅ Đã tìm thấy Image Context ID an toàn: {imgContextId}")

# ==========================================
# 4. CHUẨN BỊ DỮ LIỆU SLAKE
# ==========================================
print(f"\n--- CHUẨN BỊ DỮ LIỆU SLAKE ENGLISH (NEAR-IID α=1.0, {NUM_CLIENTS} CLIENTS) ---")

if not os.path.exists("./imgs"):
    print(" [Hệ thống] Đang tải file ảnh imgs.zip từ Hugging Face...")
    urllib.request.urlretrieve("https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip", "imgs.zip")
    with zipfile.ZipFile("imgs.zip", 'r') as zipRef:
        zipRef.extractall("./")
    os.remove("imgs.zip")
    print(" [Hệ thống] Đã chuẩn bị xong thư mục ảnh cục bộ!")

def loadSlakeImage(imgName):
    imgPath = os.path.join(".", "imgs", imgName.replace("imgs/", ""))
    return Image.open(imgPath).convert("RGB")

dataset = load_dataset("BoKelvin/SLAKE", cache_dir=dataDir)

# Lọc Tiếng Anh
englishTrain = dataset["train"].filter(lambda x: x["q_lang"] == "en")
englishTestAll = dataset["test"].filter(lambda x: x["q_lang"] == "en")
testDataset = englishTestAll.shuffle(seed=42).select(range(min(500, len(englishTestAll))))
shuffledTrain = englishTrain.shuffle(seed=42)

def partitionDataNonIidDirichlet(datasetInput, numClients, alpha=1.0):
    labels = [str(item["answer"]).strip().lower() for item in datasetInput]
    uniqueLabels = list(set(labels))
    clientIndices = {i: [] for i in range(numClients)}
    for label in uniqueLabels:
        labelIdxs = [i for i, l in enumerate(labels) if l == label]
        np.random.shuffle(labelIdxs)
        proportions = np.random.dirichlet([alpha] * numClients)
        splits = np.split(labelIdxs, (np.cumsum(proportions)[:-1] * len(labelIdxs)).astype(int))
        for i in range(numClients):
            clientIndices[i].extend(splits[i].tolist())
            
    return [datasetInput.select(np.random.permutation(clientIndices[i])) for i in range(numClients)]

clientNames = [chr(65+i) for i in range(NUM_CLIENTS)] # A, B
clientDatasets = partitionDataNonIidDirichlet(shuffledTrain, numClients=NUM_CLIENTS, alpha=1.0)
trainLoaders = []
dataSizes = [len(ds) for ds in clientDatasets]

print(f"Phân bổ dữ liệu cho {NUM_CLIENTS} Clients:")
for i, name in enumerate(clientNames):
    print(f" - Client {name}: {dataSizes[i]} mẫu")
print(f"Dữ liệu Global Test: {len(testDataset)} mẫu")

systemInstruction = (
    "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
    "based EXCLUSIVELY on the provided medical image.\nStrictly follow these rules:\n"
    "1. Analyze the provided image.\n2. Provide a concise and definitive answer (1-3 words if possible)."
)

def customCollateFn(batch):
    pixelValuesList, inputIdsList, labelsList = [], [], []
    for item in batch:
        img = loadSlakeImage(item["img_name"])
        q, a = item["question"], str(item["answer"]).strip()
        
        promptPart1 = f"{systemInstruction}\nUser: <img>"
        promptPart2 = f"</img>\nQuestion: {q}\nFinal Answer:\nAssistant: "
        
        part1Ids = tokenizer(promptPart1, return_tensors="pt", add_special_tokens=True).input_ids[0]
        part2Ids = tokenizer(promptPart2, return_tensors="pt", add_special_tokens=False).input_ids[0]
        answerIds = tokenizer(a + getattr(tokenizer, 'eos_token', ''), return_tensors="pt", add_special_tokens=False).input_ids[0]
        
        # Với ảnh 448x448, số token tương ứng là 256
        imgTokens = torch.full((256,), imgContextId, dtype=torch.long)
        inputIds = torch.cat([part1Ids, imgTokens, part2Ids, answerIds], dim=0)
        
        lbls = inputIds.clone()
        promptLen = len(part1Ids) + 256 + len(part2Ids)
        lbls[:promptLen] = -100 
        
        inputIdsList.append(inputIds)
        labelsList.append(lbls)
        pixelValuesList.append(processImage(img))

    maxLen = max(len(ids) for ids in inputIdsList)
    padId = getattr(tokenizer, 'pad_token_id', getattr(tokenizer, 'eos_token_id', 0))
    if padId is None:
        padId = 0
    
    paddedInputIds, paddedLabels, attentionMask = [], [], []
    for ids, lbls in zip(inputIdsList, labelsList):
        padLen = maxLen - len(ids)
        paddedInputIds.append(torch.cat([ids, torch.full((padLen,), padId, dtype=torch.long)]))
        paddedLabels.append(torch.cat([lbls, torch.full((padLen,), -100, dtype=torch.long)]))
        attentionMask.append(torch.cat([torch.ones(len(ids), dtype=torch.long), torch.zeros(padLen, dtype=torch.long)]))

    return {
        "input_ids": torch.stack(paddedInputIds),
        "labels": torch.stack(paddedLabels),
        "attention_mask": torch.stack(attentionMask),
        "pixel_values": torch.cat(pixelValuesList, dim=0).to(torch.float16),
        "image_flags": torch.ones((len(batch), 1), dtype=torch.long) 
    }

# ÉP BUỘC BATCH SIZE = 1 CHO FFT ĐỂ TRÁNH OOM, tắt num_workers
batchSize = 1

for ds in clientDatasets:
    trainLoaders.append(DataLoader(
        ds, 
        batch_size=batchSize, 
        shuffle=True, 
        collate_fn=customCollateFn, 
        num_workers=0, 
        pin_memory=True
    ))

# ==========================================
# 5. CÁC HÀM XỬ LÝ CHỐNG TRÀN RAM & ĐÁNH GIÁ 
# ==========================================
def trainLocalClient(model, dataLoader, clientId, roundIdx, epochs=2, learningRate=1e-5): 
    model.train() 
    
    decayFactor = 0.9 ** (roundIdx - 1)
    activeLr = learningRate * decayFactor
    
    # Phải dùng PagedAdamW8bit vì đây là FFT
    optimizer = bnb.optim.PagedAdamW8bit(model.parameters(), lr=activeLr, weight_decay=0.01)
    
    # Do batch=1 nên ta dùng accumulate=4 để bù đắp quỹ đạo gradient
    accumulationSteps = 4 
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs * len(dataLoader) // accumulationSteps)
    
    print(f" [Client {clientId}] Bắt đầu huấn luyện FFT (PagedAdamW8bit, LR={activeLr:.2e})...")
    startTime = time.time()
    totalLoss = 0
    
    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)
        progressLoop = tqdm(dataLoader, desc=f"Client {clientId} | Epoch {epoch+1}/{epochs}", leave=False, dynamic_ncols=True)
        for step, batch in enumerate(progressLoop):
            batch = {k: v.to(model.device, non_blocking=True) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            
            with torch.amp.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss / accumulationSteps
            loss.backward()
            
            if (step + 1) % accumulationSteps == 0 or (step + 1) == len(dataLoader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                
            totalLoss += (loss.item() * accumulationSteps)
            progressLoop.set_postfix(Loss=f"{(loss.item() * accumulationSteps):.4f}")
            
            del batch, loss
            
    trainDuration = time.time() - startTime
    avgLoss = totalLoss / (len(dataLoader) * epochs)
    peakVram = getPeakReservedMemory()
    
    del optimizer, scheduler
    clearMemory()
    
    # LƯU TRỌNG SỐ TRỰC TIẾP XUỐNG ĐĨA
    savePath = os.path.join(TEMP_WEIGHT_DIR, f"client_{clientId}_weights.pt")
    torch.save(model.state_dict(), savePath)
    clearMemory()
    
    return savePath, trainDuration, avgLoss, peakVram

# TỔNG HỢP THEO TỪNG LAYER ĐỂ CHỐNG OOM CPU RAM
def diskBasedFederatedAveraging(baselinePathInput, clientWeightPaths, dataSizesInput):
    print(" Đang tiến hành Tối ưu hóa Disk-Streaming Layer-by-Layer (Chống tràn RAM CPU tuyệt đối)...")
    totalSamples = sum(dataSizesInput)
    
    baselineState = torch.load(baselinePathInput, map_location="cpu", mmap=True, weights_only=True)
    layerKeys = list(baselineState.keys())
    
    newGlobalWeights = {}
    
    for key in layerKeys:
        if not baselineState[key].dtype.is_floating_point:
            newGlobalWeights[key] = baselineState[key].clone()
            continue
            
        layerSum = torch.zeros_like(baselineState[key], device="cpu", dtype=torch.float32)
        
        for i, path in enumerate(clientWeightPaths):
            weightRatio = dataSizesInput[i] / totalSamples
            clientState = torch.load(path, map_location="cpu", mmap=True, weights_only=True)
            
            layerSum.add_(clientState[key].float() * weightRatio)
            del clientState
        
        newGlobalWeights[key] = layerSum.to(torch.float16)
        
        del layerSum
        gc.collect()

    del baselineState
    clearMemory()
    
    for path in clientWeightPaths:
        if os.path.exists(path):
            os.remove(path)
            
    print(" ✅ Hoàn tất tổng hợp FedAvg. RAM CPU an toàn!")
    return newGlobalWeights

def cleanOpenAnswer(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    stopwords = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'it', 'there', 'shows', 'showing', 'of', 'in', 'on', 'to'}
    words = text.split()
    words = [w for w in words if w not in stopwords]
    return " ".join(words)

def evaluateGlobalModel(model, datasetInput):
    model.eval()
    print(f"\n [Server] Đang đánh giá trên tập Test ({len(datasetInput)} mẫu)...")
    
    openPreds, openRefs, closePreds, closeRefs, inferenceTimes = [], [], [], [], []
    overallCorrect = 0
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    evalLoop = tqdm(datasetInput, desc="Evaluating", leave=False, dynamic_ncols=True)
    
    for item in evalLoop:
        q = item["question"]
        gtAnswer = str(item["answer"]).strip().lower()
        ansType = str(item.get("answer_type", "")).strip().upper()
        if ansType == "":
            ansType = "CLOSED" if gtAnswer in ["yes", "no"] else "OPEN"
        
        promptText = f"{systemInstruction}\n[Target Case Image]\n<image>\n\nCurrent Question: {q}\nBased on the visual evidence, what is the correct answer?\nFinal Answer: "
        img = loadSlakeImage(item["img_name"])
        pixelValues = processImage(img).to(torch.float16).cuda()
        
        startTime = time.time()
        with torch.inference_mode():
            aiAnswerRaw = model.chat(
                tokenizer=tokenizer,
                pixel_values=pixelValues,
                question=promptText,
                generation_config=dict(max_new_tokens=15, do_sample=False)
            )
        inferenceTimes.append(time.time() - startTime)
        
        aiAnswer = aiAnswerRaw.lower().replace("final answer:", "").strip()
        aiAnswerClean = cleanOpenAnswer(aiAnswer)
        gtAnswerClean = cleanOpenAnswer(gtAnswer)
        
        if ansType == "CLOSED":
            if gtAnswer in ['yes', 'no']:
                hasYes = any(word in aiAnswer for word in yesKeywords)
                hasNo = any(word in aiAnswer for word in noKeywords)
                if hasYes and not hasNo:
                    predClean = 'yes'
                elif hasNo and not hasYes:
                    predClean = 'no'
                else:
                    firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                    firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                    predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
                targetGt = gtAnswer
            else:
                predClean = aiAnswerClean if aiAnswerClean else "unknown"
                targetGt = gtAnswerClean if gtAnswerClean else gtAnswer
                
            closePreds.append(predClean)
            closeRefs.append(targetGt)
            
            if predClean == targetGt:
                overallCorrect += 1
        else:
            predOpen = aiAnswerClean if aiAnswerClean else "none"
            refOpen = gtAnswerClean if gtAnswerClean else "none"
            openPreds.append(predOpen)
            openRefs.append(refOpen)
            
            if predOpen == refOpen:
                overallCorrect += 1
            
        del pixelValues
        torch.cuda.empty_cache()
        
    res = {
        "avgInferenceTime": np.mean(inferenceTimes) if inferenceTimes else 0,
        "overallAccuracy": overallCorrect / len(datasetInput) if len(datasetInput) > 0 else 0
    }
    if closeRefs:
        res["Closed"] = {
            "ACC": accuracy_score(closeRefs, closePreds),
            "F1": f1_score(closeRefs, closePreds, average='weighted', zero_division=0),
            "count": len(closeRefs)
        }
    if openRefs:
        bertRes = evaluate.load("bertscore").compute(predictions=openPreds, references=openRefs, lang="en")
        res["Open"] = {
            "BLEU-1": evaluate.load("bleu").compute(predictions=openPreds, references=[[r] for r in openRefs], max_order=1)["bleu"],
            "ROUGE-L": evaluate.load("rouge").compute(predictions=openPreds, references=[[r] for r in openRefs])["rougeL"],
            "BERTScore": np.mean(bertRes["f1"]),
            "count": len(openRefs)
        }
    gc.collect()
    return res

# ==========================================
# 6. VÒNG LẶP HỌC LIÊN KẾT CHÍNH (FED-AVG, FFT)
# ==========================================
maxRounds = 1
localEpochs = 2
performanceLog = []
cumulativeCommGb = 0.0

print(f"\n=== BẮT ĐẦU HUẤN LUYỆN FED-FFT INTERNVL3.5-2B (SLAKE, {maxRounds} ROUNDS, {localEpochs} EPOCHS/CLIENT, ALPHA=1.0) ===")
for roundIdx in range(1, maxRounds + 1):
    print(f"\n" + "="*50)
    print(f"--- COMMUNICATION ROUND {roundIdx} ---")
    print("="*50)
    
    latestTrainTimes, localWeightPaths, roundLoss = [], [], 0
    peakVramRecorded = "0.00 GB"
    
    # 6.1 Huấn luyện từng Client
    for i, clientId in enumerate(clientNames):
        baseModel.load_state_dict(torch.load(baselinePath, map_location="cpu", weights_only=True), strict=False)
        clearMemory()
        
        savedPath, t, l, vram = trainLocalClient(baseModel, trainLoaders[i], clientId, roundIdx=roundIdx, epochs=localEpochs, learningRate=1e-5)
        localWeightPaths.append(savedPath)
        latestTrainTimes.append(t)
        roundLoss += l
        peakVramRecorded = vram
        clearMemory()
    
    # 6.2 Tổng hợp trọng số qua Ổ Cứng (Disk Streaming LAYER-BY-LAYER CHỐNG OOM)
    aggregatedWeights = diskBasedFederatedAveraging(baselinePath, localWeightPaths, dataSizes)
    torch.save(aggregatedWeights, baselinePath)
    
    baseModel.load_state_dict(aggregatedWeights, strict=False)
    del aggregatedWeights
    clearMemory()
    
    # 6.3 Đánh giá
    evalRes = evaluateGlobalModel(baseModel, testDataset)
    
    # 6.4 Bóc tách các Metric chi tiết
    roundAccuracy = evalRes["overallAccuracy"] * 100
    roundAccClosed = evalRes.get("Closed", {}).get("ACC", 0) * 100
    roundF1Closed = evalRes.get("Closed", {}).get("F1", 0) * 100
    roundBleu = evalRes.get("Open", {}).get("BLEU-1", 0)
    roundRouge = evalRes.get("Open", {}).get("ROUGE-L", 0)
    roundBert = evalRes.get("Open", {}).get("BERTScore", 0)
    
    fullRoundMb = calculateCommunicationMetrics(totalModelParams, NUM_CLIENTS)
    cumulativeCommGb += (fullRoundMb / 1024)
    avgTrainingTime = np.mean(latestTrainTimes)
    averageRoundLoss = roundLoss / NUM_CLIENTS
    
    roundLog = {
        "Round": roundIdx,
        "Avg Loss": round(averageRoundLoss, 4),
        "Overall Acc (%)": round(roundAccuracy, 2),
        "Closed Acc (%)": round(roundAccClosed, 2),
        "Closed F1 (%)": round(roundF1Closed, 2),
        "Open BLEU-1": round(roundBleu, 4),
        "Open ROUGE-L": round(roundRouge, 4),
        "Open BERTScore": round(roundBert, 4),
        "MB per Round": round(fullRoundMb, 2),
        "Cumul Data (GB)": round(cumulativeCommGb, 4),
        "Avg Train Time (s)": round(avgTrainingTime, 2)
    }
    performanceLog.append(roundLog)
    pd.DataFrame(performanceLog).to_csv(logPath, index=False)
    
    print(f"\n[VÒNG {roundIdx}] Loss: {averageRoundLoss:.4f} | Acc: {roundAccuracy:.2f}% | Closed Acc: {roundAccClosed:.2f}%, F1: {roundF1Closed:.2f}% | Open BLEU-1: {roundBleu:.4f}, ROUGE-L: {roundRouge:.4f} | VRAM: {peakVramRecorded}")

# ==========================================
# 7. BÁO CÁO TỔNG KẾT CHI TIẾT
# ==========================================
print("\n" + "="*80)
print(f" FINAL SUMMARY REPORT: FED-FFT INTERNVL3.5 (SLAKE, {maxRounds} ROUNDS, 2 EPOCHS, {NUM_CLIENTS} CLIENTS, ALPHA=1.0) ")
print("="*80)

print("\n[1] PERFORMANCE & HARDWARE METRICS")
print(f"  + Trainable Parameters   : {trainableParams:,} (~100%)")
print(f"  + Peak Reserved Memory   : {peakVramRecorded}")
print(f"  + Average Inference Time : {evalRes['avgInferenceTime']:.4f} giây/ảnh")

print("\n[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)")
if "Closed" in evalRes:
    print(f"  - CLOSED QUESTIONS ({evalRes['Closed'].get('count', 'N/A')} samples):")
    print(f"    * Accuracy : {evalRes['Closed']['ACC']*100:.2f}%")
    print(f"    * F1-Score : {evalRes['Closed']['F1']*100:.2f}%")
if "Open" in evalRes:
    print(f"  - OPEN QUESTIONS ({evalRes['Open'].get('count', 'N/A')} samples):")
    print(f"    * BLEU-1   : {evalRes['Open']['BLEU-1']:.4f}")
    print(f"    * ROUGE-L  : {evalRes['Open']['ROUGE-L']:.4f}")
    print(f"    * BERTScore: {evalRes['Open']['BERTScore']:.4f}")

print("\n[3] CONVERGENCE LOG CHI TIẾT")
commTable = pd.DataFrame(performanceLog)
print(commTable[['Round', 'Avg Loss', 'Overall Acc (%)', 'Closed Acc (%)', 'Open BLEU-1', 'MB per Round', 'Cumul Data (GB)', 'Avg Train Time (s)']].to_string(index=False))

print("\n" + "="*80)
print(" QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG ")
print("="*80)

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


--- KHỞI TẠO HẠ TẦNG FED-FFT (INTERNVL3.5-2B, 448x448, FP16 - 2 CLIENTS) ---


config.json:   0%|          | 0.00/5.75k [00:00<?, ?B/s]

configuration_internvl_chat.py:   0%|          | 0.00/4.70k [00:00<?, ?B/s]

configuration_intern_vit.py:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

modeling_internvl_chat.py:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

... (35 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]


[VÒNG 1] Loss: 0.3017 | Acc: 79.20% | Closed Acc: 89.83%, F1: 89.36% | Open BLEU-1: 0.7154, ROUGE-L: 0.7984 | VRAM: GPU0: 5.7GB | GPU1: 5.5GB

 FINAL SUMMARY REPORT: FED-FFT INTERNVL3.5 (SLAKE, 1 ROUNDS, 2 EPOCHS, 2 CLIENTS, ALPHA=1.0) 

[1] PERFORMANCE & HARDWARE METRICS
  + Trainable Parameters   : 2,348,347,392 (~100%)
  + Peak Reserved Memory   : GPU0: 5.7GB | GPU1: 5.5GB
  + Average Inference Time : 0.2808 giây/ảnh

[2] CLINICAL EVALUATION METRICS (GLOBAL MODEL VÒNG CUỐI)
  - CLOSED QUESTIONS (177 samples):
    * Accuracy : 89.83%
    * F1-Score : 89.36%
  - OPEN QUESTIONS (323 samples):
    * BLEU-1   : 0.7154
    * ROUGE-L  : 0.7984
    * BERTScore: 0.9745

[3] CONVERGENCE LOG CHI TIẾT
 Round  Avg Loss  Overall Acc (%)  Closed Acc (%)  Open BLEU-1  MB per Round  Cumul Data (GB)  Avg Train Time (s)
     1    0.3017             79.2           89.83       0.7154      17916.47          17.4966             4394.01

 QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT THÀNH CÔNG 


In [ ]:
import os
os.system('kill -9 -1')